<a href="https://colab.research.google.com/github/ALX-Courses/alura-dev-agents-ia-google/blob/main/colab/alura_agent_ia_google.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 1 - Intent Classification (Triage)

In [ ]:
!pip install -q --upgrade langchain langchain-google-genai google-generativeai

In [ ]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
llm = ChatGoogleGenerativeAI(
   model="gemini-2.5-flash",
   # Level of creativity - 0..1
   temperature=0.0,
   api_key=GOOGLE_API_KEY
)

In [ ]:
answer = llm.invoke("Quem é você?")
print(answer.content)

In [ ]:
triage_prompt = (
    "Você é um triador de Service Desk para políticas internas da empresa Carraro Desenvolvimento. "
    "Dada a mensagem do usuário, retorne SOMENTE um JSON com:\n"
    "{\n"
    '  "decisao": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_CHAMADO",\n'
    '  "urgencia": "BAIXA" | "MEDIA" | "ALTA",\n'
    '  "campos_faltantes": ["..."]\n'
    "}\n"
    "Regras:\n"
    '- **AUTO_RESOLVER**: Perguntas claras sobre regras ou procedimentos descritos nas políticas (Ex: "Posso reembolsar a internet do meu home office?", "Como funciona a política de alimentação em viagens?").\n'
    '- **PEDIR_INFO**: Mensagens vagas ou que faltam informações para identificar o tema ou contexto (Ex: "Preciso de ajuda com uma política", "Tenho uma dúvida geral").\n'
    '- **ABRIR_CHAMADO**: Pedidos de exceção, liberação, aprovação ou acesso especial, ou quando o usuário explicitamente pede para abrir um chamado (Ex: "Quero exceção para trabalhar 5 dias remoto.", "Solicito liberação para anexos externos.", "Por favor, abra um chamado para o RH.").'
    "Analise a mensagem e decida a ação mais apropriada."
)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, List, Dict

class TriageOut(BaseModel):
  decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_CHAMADO"]
  urgency: Literal["BAIXA", "MEDIA", "ALTA"]
  missing_fields: List[str] = Field(default_factory=list)

In [ ]:
llm_triage = ChatGoogleGenerativeAI(
   model="gemini-2.5-flash",
   # Level of creativity - 0..1
   temperature=0.0,
   api_key=GOOGLE_API_KEY
)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

triage_chain = llm_triage.with_structured_output(TriageOut)

def triage(message: str) -> Dict:
  output: TriageOut = triage_chain.invoke(
    [
      SystemMessage(content=triage_prompt),
      HumanMessage(content=message)
    ]
  )

  return output.model_dump()

In [ ]:
tests = ["Posso reembolsar a internet?",
         "Quero mais 5 de trabalho remoto. Como faço?",
         "Posso reembolsar cursos ou treinamentos da Alura?",
         "Quantas capivaras tem no Rio Pinheiros?"]

In [ ]:
for msg_test in tests:
  print(f"Pergunta: {msg_test}\n -> Resposta {triage(msg_test)}\n")

# Lesson 2 - Building a Knowledge Base with RAG

In [ ]:
!pip install -q --upgrade langchain_community faiss-cpu langchain-text-splitters pymupdf

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for file in Path("/content/").glob("*.pdf"):
  try:
    loader = PyMuPDFLoader(str(file))
    docs.extend(loader.load())
    print(f"File uploaded successfully: {file.name}")
  except Exception as e:
    print(f"File upload failed: {file.name}")

print(f"Total documents uploaded: {len(docs)}")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30)

chunks = splitter.split_documents(docs)

In [ ]:
for chunk in chunks:
  print(chunk)
  print("---")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold":0.3, "k": 4}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um Assistente de Políticas Internas (RH/IT) da empresa Carraro Desenvolvimento. "
     "Responda SOMENTE com base no contexto fornecido. "
     "Se não houver base suficiente, responda apenas 'Não sei'."),

    ("human", "Pergunta: {input}\n\nContexto:\n{context}")
])

document_chain = create_stuff_documents_chain(llm_screeaning, rag_prompt)

In [ ]:
def ask_rag(question: str) -> Dict:
  docs_related = retriever.invoke(question)

  if not docs_related:
    return {"answer": "Não sei.",
            "quotes": [],
            "found_context": False}

  answer = document_chain.invoke({"input": question,
                                  "context": docs_related})

  text = (answer or "").strip()

  if text.rstrip(".!?") == "Não sei":
    return {"answer": "Não sei.",
            "quotes": [],
            "found_context": False}

  return {"answer": text,
          "quotes": docs_related,
          "found_context": True}

In [ ]:
tests = ["Posso reembolsar a internet?",
         "Quero mais 5 de trabalho remoto. Como faço?",
         "Posso reembolsar cursos ou treinamentos da Alura?",
         "Quantas capivaras tem no Rio Pinheiros?"]

In [ ]:
for msg_test in tests:
  answer = ask_rag(msg_test)
  print(f"Pergunta: {msg_test}")
  print(f"Resposta: {answer['answer']}")
  print(f"Citações: {answer['quotes']}")

# Lesson 3 - Agent Orchestration with LangGraph

In [ ]:
!pip install -q --upgrade langgraph

In [ ]:
from typing import TypedDict, Optional

class AgentState(TypedDict, total = False):
  question: str
  triage: Dict
  answer: Optional[str]
  quotes: List[dict]
  rag_success: bool
  action: str

In [ ]:
def node_triage(state: AgentState) -> AgentState:
  print("Running the Triade Node...")

  return {"triage": triage(state["question"])}

In [ ]:
def node_auto_resolve(state: AgentState) -> AgentState:
  print("Running the Auto Resolve Node...")

  answer_rag = ask_rag(state["question"])

  update: AgentState = {
      "answer": answer_rag["answer"],
      "quotes": answer_rag.get("quotes", []),
      "rag_success": answer_rag["found_context"]
  }

  if answer_rag["found_context"]:
    update["action"] = "AUTO_RESOLVER"

  return update

In [ ]:
def node_request_info(state: AgentState) -> AgentState:
  print("Running the Request Info Node...")

  missing = state["triage"].get("missing_fields", [])

  # detail = ",".join(missing) if missing else "Tema e contexto específico"
  if missing:
    detail = ",".join(missing)
  else:
    detail = "Tema e contexto específico"

  return {
      "answer": f"Para avançar, preciso que detalhe: {detail}",
      "quotes": [],
      "action": "PEDIR_INFO"
  }

In [ ]:
def node_open_ticket(state: AgentState) -> AgentState:
  print("Running the Open Ticket Node...")

  triage = state["triage"]

  return {
      "answer": f"Abrindo chamado com urgência {triage['urgency']}. Descrição: {state['question'][:140]}",
      "quotes": [],
      "action": "ABRIR_CHAMADO"
  }

In [ ]:
KEYWORDS_OPEN_TICKET = ["aprovação", "exceção", "liberação", "abrir ticket", "abrir chamado", "acesso especial"]

In [ ]:
def pos_triage_edge(state: AgentState) -> str:
  print("Deciding after triage...")

  decision = state["triage"]["decision"]

  if decision == "AUTO_RESOLVER":
    return "auto"
  elif decision == "PEDIR_INFO":
    return "info"
  else:
    return "ticket"


In [ ]:
def pos_auto_resolve_edge(state: AgentState) -> str:
  print("Deciding after auto-resolve...")

  if state.get("rag_success"):
    print("Rag com sucesso, finalizando o fluxo.")
    return "ok"

  question_state = (state["question"] or "").lower()

  if any(k in question_state for k in KEYWORDS_OPEN_TICKET):
    print("Rag falhou, mas foram encontradas keywords de abertura de ticket. Abrindo...")
    return "ticket"

  print("Rag falhou, sem keywords. Pedindo mais informações...")
  return "info"

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)

workflow.add_node("triage", node_triage)
workflow.add_node("auto_resolve", node_auto_resolve)
workflow.add_node("request_info", node_request_info)
workflow.add_node("open_ticket", node_open_ticket)

workflow.add_edge(START, "triage")
workflow.add_conditional_edges("triage", pos_triage_edge, {
    "auto": "auto_resolve",
    "info": "request_info",
    "ticket": "open_ticket"})
workflow.add_conditional_edges("auto_resolve", pos_auto_resolve_edge, {
    "info": "request_info",
    "ticket": "open_ticket",
    "ok": END})
workflow.add_edge("request_info", END)
workflow.add_edge("open_ticket", END)

graph = workflow.compile()


In [ ]:
from IPython.display import display, Image

graph_bytes= graph.get_graph().draw_mermaid_png()
display(Image(graph_bytes))

In [ ]:
tests = ["Posso reembolsar a internet?",
         "Quero mais 5 de trabalho remoto. Como faço?",
         "Posso reembolsar cursos ou treinamentos da Alura?",
         "É possível reembolsar certificações do Google Cloud?",
         "Posso obter o Google Gemini de graça?",
         "Qual é a palavra-chave da última aula de imersão?",
         "Quantas capivaras tem no Rio Pinheiros?"]

In [ ]:
for msg_test in tests:
  final_answer = graph.invoke({"question": msg_test})

  triage_test = final_answer.get("triage", {})
  print(f"Pergunta: {msg_test}")
  print(f"Decisão: {triage_test.get('decision')}, Urgência: {triage_test.get('urgency')}, Ação: {final_answer.get('action')}")
  print(f"Resposta: {final_answer.get('answer')}")
  print("-----------------------------------")